# 👥 Demographic Data Analysis
**Source:** [DummyJSON API](https://dummyjson.com/users)  
**Tools:** Python, Pandas, Seaborn, Matplotlib

## 1. Load Data
Importing all required libraries and loading the dataset from the CSV file that was fetched from the API.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

df = pd.read_csv('users.csv')
print(f'✔ Loaded {len(df)} rows, {len(df.columns)} columns')
df.head()

## 2. Basic Data Exploration

### 📐 Shape
Displays the number of rows and columns in the dataset.

In [ ]:
print('Shape:', df.shape)

### 📋 Column Names
Lists all available columns in the dataset.

In [ ]:
print(list(df.columns))

### 🔠 Data Types
Shows the data type of each column (int, float, object, etc.).

In [ ]:
df.dtypes

### ❓ Missing Values
Counts the number of null/missing values in each column.

In [ ]:
df.isnull().sum()

### ♻️ Duplicate Rows
Checks if there are any duplicate rows in the dataset.

In [ ]:
print(f'Duplicate Rows: {df.duplicated().sum()}')

### 📊 Summary Statistics
Shows statistical summary for all numeric columns (mean, std, min, max, etc.).

In [ ]:
df.describe().round(2)

### 📂 Value Counts — Categorical Columns
Shows how many times each unique value appears in important categorical columns like gender, bloodGroup, eyeColor, and role.

In [ ]:
for col in ['gender', 'bloodGroup', 'eyeColor', 'role']:
    if col in df.columns:
        print(f'\n[{col}]')
        print(df[col].value_counts())

## 3. Data Cleaning

### 🏙️ Extract City & Country from Address
The address column contains nested data (like a dictionary). We extract the city and country from it using a custom parsing function.

In [ ]:
def parse_address(addr_str, key):
    try:
        addr_dict = json.loads(str(addr_str).replace("'", '"'))
        return addr_dict.get(key, 'Unknown')
    except:
        try:
            return eval(str(addr_str)).get(key, 'Unknown')
        except:
            return 'Unknown'

df['city']    = df['address'].apply(lambda x: parse_address(x, 'city'))
df['country'] = df['address'].apply(lambda x: parse_address(x, 'country'))

print('✔ city & country extracted')
df[['city', 'country']].head()

### 🩹 Handle Missing Values
Fills any missing values in age, height, and weight columns with the median value of each column.

In [ ]:
for col in ['age', 'height', 'weight']:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)

print('✔ Missing values handled')
print(df[['age','height','weight']].isnull().sum())

### 🌍 Country Value Counts
Shows how many users exist per country after extracting the country from the address.

In [ ]:
df['country'].value_counts()

## 4. Analysis

### Q1 — Average Age of Users
Calculates the mean age across all users in the dataset.

In [ ]:
avg_age = df['age'].mean()
print(f'Average age of users: {avg_age:.2f} years')

### Q2 — Average Age by Gender
Groups users by gender and calculates the average age for each group.

In [ ]:
df.groupby('gender')['age'].mean().round(2)

### Q3 — Number of Users per Gender
Counts how many users belong to each gender category.

In [ ]:
df['gender'].value_counts()

### Q4 — Top 10 Cities with Most Users
Finds the 10 cities that appear most frequently in the dataset.

In [ ]:
df['city'].value_counts().head(10)

### Q5 — Average Height and Weight
Calculates the overall average height (cm) and weight (kg) across all users.

In [ ]:
print(f'Average height: {df["height"].mean():.2f} cm')
print(f'Average weight: {df["weight"].mean():.2f} kg')

### Q6 — Relationship Between Age and Height/Weight
Uses Pearson correlation to measure the linear relationship between age and both height and weight. A value close to 0 means weak or no relationship.

In [ ]:
corr_h = df[['age','height']].corr().loc['age','height']
corr_w = df[['age','weight']].corr().loc['age','weight']
print(f'Pearson r — age ↔ height : {corr_h:.4f}')
print(f'Pearson r — age ↔ weight : {corr_w:.4f}')
print('→ Weak correlation: no strong relationship between age and height/weight.')

## 5. Visualizations

### Plot 1 — Age Distribution
Histogram with KDE curve showing how user ages are distributed. The red dashed line marks the mean age.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['age'], bins=20, kde=True, color='steelblue')
plt.axvline(avg_age, color='crimson', linestyle='--', label=f'Mean: {avg_age:.1f}')
plt.title('Distribution of User Ages', fontsize=14, fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

### Plot 2 — Average Age by Gender
Bar chart comparing the average age between different gender groups.

In [ ]:
avg_age_gender = df.groupby('gender')['age'].mean().round(2)
plt.figure(figsize=(6,5))
ax = avg_age_gender.plot(kind='bar', color=['#4C72B0','#DD8452'], edgecolor='white')
plt.title('Average Age by Gender', fontsize=14, fontweight='bold')
plt.xlabel('Gender')
plt.ylabel('Average Age')
plt.xticks(rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}', (p.get_x()+p.get_width()/2, p.get_height()+0.2), ha='center')
plt.tight_layout()
plt.show()

### Plot 3 — Users by Gender (Pie Chart)
Pie chart showing the percentage of users belonging to each gender.

In [ ]:
users_gender = df['gender'].value_counts()
plt.figure(figsize=(6,6))
plt.pie(users_gender, labels=users_gender.index, autopct='%1.1f%%',
        colors=['#4C72B0','#DD8452','#55A868'],
        startangle=140, wedgeprops=dict(edgecolor='white'))
plt.title('User Count by Gender', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Plot 4 — Top 10 Cities
Horizontal bar chart showing the 10 cities with the highest number of users.

In [ ]:
top_cities = df['city'].value_counts().head(10)
plt.figure(figsize=(9,6))
sns.barplot(x=top_cities.values, y=top_cities.index, palette='Blues_r')
plt.title('Top 10 Cities by User Count', fontsize=14, fontweight='bold')
plt.xlabel('Number of Users')
plt.ylabel('City')
plt.tight_layout()
plt.show()

### Plot 5 — Age vs Height
Scatter plot with regression line showing the relationship between age and height.

In [ ]:
plt.figure(figsize=(8,5))
sns.regplot(data=df, x='age', y='height',
            scatter_kws={'alpha':0.5,'color':'#4C72B0'},
            line_kws={'color':'crimson'})
plt.title(f'Age vs Height  (r = {corr_h:.3f})', fontsize=14, fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Height (cm)')
plt.tight_layout()
plt.show()

### Plot 6 — Age vs Weight
Scatter plot with regression line showing the relationship between age and weight.

In [ ]:
plt.figure(figsize=(8,5))
sns.regplot(data=df, x='age', y='weight',
            scatter_kws={'alpha':0.5,'color':'#DD8452'},
            line_kws={'color':'steelblue'})
plt.title(f'Age vs Weight  (r = {corr_w:.3f})', fontsize=14, fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Weight (kg)')
plt.tight_layout()
plt.show()

### Plot 7 — Correlation Heatmap
Heatmap displaying the Pearson correlation between age, height, and weight. Values close to 0 indicate weak relationships.

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(df[['age','height','weight']].corr(),
            annot=True, fmt='.3f', cmap='coolwarm',
            linewidths=0.5, square=True, annot_kws={'size':12})
plt.title('Correlation — Age, Height, Weight', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()